# Demos: 12 Scikit-Learn Essentials

Each cell below is one demo from the slides; the heading matches the pill on the slide. Run the first cell, then run the demos in any order with Shift+Enter. The same code is in the `.py` files in this folder.

In [ ]:
# Run this cell first: the demos read their data files from the working folder.
import os
from pathlib import Path

COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir())
os.chdir(COURSE / "datasets/demos/12-scikit-learn")
print("Working folder:", Path.cwd())

## demo01 · Estimator API

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

auto = pd.read_csv("auto-mpg.csv")
X = auto[["weight"]]
y = auto["mpg"]

model = LinearRegression()
model.fit(X, y)
print(model.coef_, model.intercept_)

new_cars = pd.DataFrame({"weight": [2500, 4000]})
print(model.predict(new_cars).round(1))

## demo02 · Train and test

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

auto = pd.read_csv("auto-mpg.csv")
X = auto[["weight", "hp", "yr"]]
y = auto["mpg"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), len(X_test))

model = LinearRegression().fit(X_train, y_train)
predicted = model.predict(X_test)
print(round(mean_absolute_error(y_test, predicted), 2), round(r2_score(y_test, predicted), 3))

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(y_test, predicted, s=12, alpha=0.7)
ax.plot([10, 45], [10, 45], color="black", linestyle="--", linewidth=1)
ax.set(title="Test cars: predicted against actual", xlabel="Actual mpg", ylabel="Predicted mpg")
fig.tight_layout()
fig.savefig("predicted-vs-actual.png")
plt.show()

## demo03 · Logistic regression

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

titanic = pd.read_csv("titanic.csv").dropna(subset=["age", "fare"])
X = titanic[["pclass", "age", "fare"]].assign(male=titanic["sex"].eq("male").astype(int))
y = titanic["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
predicted = model.predict(X_test)
print(round(accuracy_score(y_test, predicted), 3))
print(round(1 - y_test.mean(), 3))
print(confusion_matrix(y_test, predicted))
print(model.predict_proba(X_test.head(3)).round(2))

## demo04 · Pipeline

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

titanic = pd.read_csv("titanic.csv")
X = titanic[["pclass", "sex", "age", "fare", "embarked"]]
y = titanic["survived"]

prepare = ColumnTransformer(
    [
        (
            "numbers",
            make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
            ["pclass", "age", "fare"],
        ),
        (
            "categories",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
            ),
            ["sex", "embarked"],
        ),
    ]
)
model = make_pipeline(prepare, LogisticRegression(max_iter=1000))
print(cross_val_score(model, X, y, cv=5).round(3))
folds = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=folds)
print(scores.round(3), round(scores.mean(), 3))

## demo05 · KMeans

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

auto = pd.read_csv("auto-mpg.csv")
features = StandardScaler().fit_transform(auto[["weight", "mpg"]])
auto["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(features)
print(auto.groupby(["cluster", "origin"]).size().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for cluster, group in auto.groupby("cluster"):
    ax.scatter(group["weight"], group["mpg"], s=12, label=f"cluster {cluster}")
ax.set(
    title="Three groups of cars, found without labels",
    xlabel="Weight (lb)",
    ylabel="Miles per gallon",
)
ax.legend()
fig.tight_layout()
fig.savefig("kmeans.png")
plt.show()